# 02 — CBraMod PhysioNet-MI on Northeastern Explorer

Same reproduction as the Colab version: the **official CBraMod code** (pinned commit), the authors' preprocessing, LMDB format, 70/19/20 subject split, training defaults and pretrained checkpoint. Only *where files live and how things run* changed:

| Colab version | Explorer version |
|---|---|
| Google Drive `MyDrive/CBraMod-MI` | course folder `/courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI` |
| copy Drive → `/content` every session | no copying; train straight from the course folder |
| `pip install` inside the notebook | packages installed **once** by `hpc/setup_env.sh` |
| symlinks under `/data/datasets/...` (needs root) | copy of the authors' preprocessing script with **only that path string** replaced |
| DataLoader default of 16 workers | workers = CPUs Slurm gave the session |

**Before the first run:** `hpc/setup_env.sh` has been run once, and the PhysioNet ZIP sits in `CBraMod-MI/` (or edit `RAW_DATA` below).

**How to run** (from the repo folder):
- **First run, CPU job (data prep):** `sbatch hpc/run_notebook_cpu.sbatch notebooks/02_CBraMod_MI_explorer.ipynb`. Sections 3–6 prepare data and weights; section 7 sees no GPU and skips training.
- **Then a GPU job (training):** `sbatch hpc/run_notebook.sbatch notebooks/02_CBraMod_MI_explorer.ipynb`. Sections 3–6 skip themselves; section 7 trains.
- **Or interactively:** Open OnDemand → JupyterLab → kernel **Python (cbramod)** → *Run All* (CPU session for prep, GPU session for training).

No changes to EEG transformations, model, loss, optimizer, epochs, or evaluation.

## 1 · Paths

In [1]:
from pathlib import Path, PurePosixPath
import os, sys, subprocess, shutil, re, pickle, csv
from zipfile import ZipFile

BASE = Path('/courses/CS7150.202710/students/pitta.a/cbramod-head-interp')   # your course folder

REPO_DIR = BASE / 'cbramod-head-interp'    # your GitHub project (already cloned)
AUTHORS_REPO = BASE / 'CBraMod'            # official CBraMod code (cloned by setup_env.sh)
PROJECT = BASE / 'CBraMod-MI'              # plays the role of MyDrive/CBraMod-MI

# What rclone brought over: the PhysioNet .zip OR an already-unzipped dataset folder.
RAW_DATA = PROJECT / 'eeg-motor-movementimagery-dataset-1.0.0.zip'

DATA_DIR = PROJECT / 'data' / 'physionet_mi'
EDF_DIR = DATA_DIR / 'files'               # clean copy: files/S001/S001R04.edf ...
LMDB_DIR = DATA_DIR / 'processed_average'  # authors' LMDB
EDF_DONE = DATA_DIR / 'edf_complete.txt'
LMDB_DONE = DATA_DIR / 'lmdb_complete.txt'
WEIGHTS = PROJECT / 'models' / 'pretrained' / 'cbramod' / 'pretrained_weights.pth'
RESULTS = PROJECT / 'results' / 'physionet_mi_4class'

DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Kernel Python:', sys.executable)
print('Raw data:', RAW_DATA, '(found)' if RAW_DATA.exists() else '(not found; fine only if data is already prepared)')
print('Data:', DATA_DIR)
print('Results:', RESULTS)

Kernel Python: /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/envs/cbramod/bin/python
Raw data: /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/eeg-motor-movementimagery-dataset-1.0.0.zip (found)
Data: /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/data/physionet_mi
Results: /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/results/physionet_mi_4class


## 2 · Check the environment and the official code

Packages come from the conda env built by `hpc/setup_env.sh`; nothing is pip-installed here (on Explorer that would land in your home quota). This cell confirms the authors' repo is on the pinned commit and reports the GPU, if any.

In [2]:
OFFICIAL_COMMIT = 'b9e961003214326972c567eff390e75b0287e32a'
if not (AUTHORS_REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/wjq-learning/CBraMod.git',
                    str(AUTHORS_REPO)], check=True)
subprocess.run(['git', '-C', str(AUTHORS_REPO), 'checkout', '--quiet', OFFICIAL_COMMIT], check=True)

try:
    import lmdb, torch
    from huggingface_hub import hf_hub_download
except ImportError as e:
    raise ImportError(f'{e}. Pick the "Python (cbramod)" kernel (Kernel > Change Kernel).') from None

print("Authors' commit:", OFFICIAL_COMMIT)
print('PyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU in this session: fine for sections 3-6; section 7 needs a GPU session.')

/courses/CS7150.202710/students/pitta.a/cbramod-head-interp/envs/cbramod/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authors' commit: b9e961003214326972c567eff390e75b0287e32a
PyTorch 2.5.1+cu121 | CUDA available: True
GPU: Tesla V100-SXM2-32GB


## 3 · Build a clean EDF folder (first run only)

Same six imagery runs (04, 06, 08, 10, 12, 14) for 109 subjects = 654 files, same `files/S001/S001R04.edf` layout as on Colab. The authors' script **lists this folder to make the subject split**, so it must contain nothing but the 109 subject folders; that is why we build a clean copy instead of pointing at the raw download. Works whether `RAW_DATA` is the ZIP or an unzipped folder.

In [3]:
IMAGERY = re.compile(r'S\d{3}R(04|06|08|10|12|14)\.edf', flags=re.I)

def target_for(name):
    filename = name[:7].upper() + '.edf'           # S001R04.edf
    return EDF_DIR / filename[:4] / filename        # files/S001/S001R04.edf

if LMDB_DONE.is_file() and (LMDB_DIR / 'data.mdb').is_file():
    print('Processed LMDB already exists; EDF step not needed.')
elif EDF_DONE.is_file():
    print('EDF folder already prepared:', EDF_DIR)
else:
    if not RAW_DATA.exists():
        raise FileNotFoundError(f'Set RAW_DATA in section 1; nothing at {RAW_DATA}')
    print('Extracting the six imagery runs (one-time operation)...')
    if RAW_DATA.suffix.lower() == '.zip':
        with ZipFile(RAW_DATA) as archive:
            members = [i for i in archive.infolist()
                       if IMAGERY.fullmatch(PurePosixPath(i.filename).name)]
            assert len(members) == 654, f'Expected 654 imagery EDF recordings, found {len(members)}'
            for info in members:
                dst = target_for(PurePosixPath(info.filename).name)
                dst.parent.mkdir(parents=True, exist_ok=True)
                if dst.is_file() and dst.stat().st_size == info.file_size:
                    continue
                with archive.open(info) as source, dst.open('wb') as output:
                    shutil.copyfileobj(source, output)
    else:
        sources = [p for p in RAW_DATA.rglob('*') if IMAGERY.fullmatch(p.name)]
        assert len(sources) == 654, f'Expected 654 imagery EDF recordings, found {len(sources)}'
        for src in sources:
            dst = target_for(src.name)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    EDF_DONE.write_text('654 imagery EDF recordings extracted from original PhysioNet-MI download\n')
    print('Saved:', EDF_DIR)

Processed LMDB already exists; EDF step not needed.


## 4 · Run the authors' preprocessing (first run only)

On Colab, the authors' hard-coded `/data/datasets/eeg-motor-movementimagery-dataset-1.0.0/...` paths were redirected with symlinks, which needs root. On the cluster we instead write `preprocessing_physio_hpc.py`, a copy of their script in which **only that path string** is replaced by `DATA_DIR`; the original file is untouched. Referencing, filtering, epoching, labels and the subject split are the authors' code as-is. CPU-only work, so a CPU session is the right place for it.

In [4]:
AUTHORS_PREFIX = '/data/datasets/eeg-motor-movementimagery-dataset-1.0.0'

if LMDB_DONE.is_file() and (LMDB_DIR / 'data.mdb').is_file():
    print('Using the existing preprocessed LMDB (no reprocessing).')
else:
    assert EDF_DONE.is_file(), 'Run section 3 first.'
    original = AUTHORS_REPO / 'preprocessing' / 'preprocessing_physio.py'
    source = original.read_text()
    if AUTHORS_PREFIX not in source:
        hits = '\n'.join(line for line in source.splitlines() if '/data' in line)
        raise RuntimeError("The authors' script paths changed; review these lines:\n" + hits)
    hpc_script = original.with_name('preprocessing_physio_hpc.py')
    hpc_script.write_text(source.replace(AUTHORS_PREFIX, str(DATA_DIR)))
    LMDB_DIR.mkdir(parents=True, exist_ok=True)

    preprocessing_log = DATA_DIR / 'preprocessing.log'
    print("Running the authors' preprocessing; log:", preprocessing_log, flush=True)
    with preprocessing_log.open('w') as log:
        process = subprocess.run([sys.executable, '-u', str(hpc_script.relative_to(AUTHORS_REPO))],
                                 cwd=str(AUTHORS_REPO), stdout=log, stderr=subprocess.STDOUT)
    if process.returncode != 0:
        raise RuntimeError("Authors' preprocessing failed:\n" + preprocessing_log.read_text()[-4000:])
    LMDB_DONE.write_text(f'Preprocessed by official CBraMod commit {OFFICIAL_COMMIT}\n')
    print('Saved:', LMDB_DIR)

# Confirm the authors' train / validation / test LMDB keys exist.
db = lmdb.open(str(LMDB_DIR), readonly=True, lock=False, readahead=False)
with db.begin() as tx:
    keys = pickle.loads(tx.get(b'__keys__'))
    first = pickle.loads(tx.get(keys['train'][0].encode()))
db.close()
print('Train / val / test:', *(len(keys[k]) for k in ('train', 'val', 'test')))
print('Total trials:', sum(len(v) for v in keys.values()))
print('Example EEG shape:', first['sample'].shape)

Using the existing preprocessed LMDB (no reprocessing).


Train / val / test: 6300 1734 1803
Total trials: 9837
Example EEG shape: (64, 4, 200)


## 5 · Authors' pretrained weights (one time)

In [5]:
if not WEIGHTS.is_file():
    downloaded = hf_hub_download(repo_id='weighting666/CBraMod', filename='pretrained_weights.pth')
    WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(downloaded, WEIGHTS)
print('Pretrained checkpoint:', WEIGHTS)

Pretrained checkpoint: /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/models/pretrained/cbramod/pretrained_weights.pth


## 6 · LMDB loader fix (unchanged from the Colab version)

The authors' loader opens the **same LMDB three times** (train/val/test), which can raise an environment-already-open error. This shares one read-only connection; labels, samples, model and hyperparameters are untouched.

In [6]:
loader_path = AUTHORS_REPO / 'datasets' / 'physio_dataset.py'
source = loader_path.read_text()
old = 'self.db = lmdb.open(data_dir, readonly=True, lock=False, readahead=True, meminit=False)'
new = ("if not hasattr(CustomDataset, '_shared_db'):\n"
       "            CustomDataset._shared_db = lmdb.open(data_dir, readonly=True, lock=False, readahead=True, meminit=False)\n"
       "        self.db = CustomDataset._shared_db")
if old in source:
    source = source.replace(old, new, 1)
    compile(source, str(loader_path), 'exec')
    loader_path.write_text(source)
elif new not in source:
    raise RuntimeError("The authors' dataset loader has changed; review before patching.")
print('Original loader retained; only the shared read-only LMDB connection is adjusted.')

Original loader retained; only the shared read-only LMDB connection is adjusted.


## 7 · Fine-tune with the original `finetune_main.py` (GPU job)

Same arguments as on Colab, so the script's defaults apply: **50 epochs, batch size 64, seed 3407**. Two cluster details:
- The run folder is directly in the course folder, so there is no copy-back step.
- `--num_workers` is set to the CPUs Slurm gave this session. The script's default of 16 oversubscribes a small allocation and can hit the memory cap (`courses-gpu` allows 12 GB). Workers only affect loading speed; the shuffled batch order still comes from the seeded main process.

Without a GPU this cell skips training, so a CPU-only data-prep run finishes cleanly.

In [7]:
SEED = 3407

def finetune(seed):
    num_workers = min(8, len(os.sched_getaffinity(0)))   # CPUs allocated to this job
    run_dir = RESULTS / f'seed_{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / 'gpu.txt').write_text(torch.cuda.get_device_name(0) + '\n')
    command = [
        sys.executable, '-u', 'finetune_main.py',
        '--downstream_dataset', 'PhysioNet-MI',
        '--datasets_dir', str(LMDB_DIR),
        '--num_of_classes', '4',
        '--foundation_dir', str(WEIGHTS),
        '--model_dir', str(run_dir),
        '--cuda', '0',
        '--seed', str(seed),
        '--num_workers', str(num_workers),
    ]
    print('GPU:', torch.cuda.get_device_name(0))
    print('Running official fine-tuning:', ' '.join(command), flush=True)
    log_path = run_dir / 'train.log'
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=str(AUTHORS_REPO),
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        status = process.wait()
    if status != 0:
        raise RuntimeError(f'Fine-tuning exited with code {status}; see {log_path}')

if torch.cuda.is_available():
    finetune(SEED)
else:
    print('No GPU in this job: data prep is done, training skipped.')
    print('To train: sbatch hpc/run_notebook.sbatch notebooks/02_CBraMod_MI_explorer.ipynb')

GPU: Tesla V100-SXM2-32GB
Running official fine-tuning: /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/envs/cbramod/bin/python -u finetune_main.py --downstream_dataset PhysioNet-MI --datasets_dir /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/data/physionet_mi/processed_average --num_of_classes 4 --foundation_dir /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/models/pretrained/cbramod/pretrained_weights.pth --model_dir /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/results/physionet_mi_4class/seed_3407 --cuda 0 --seed 3407 --num_workers 4


Namespace(seed=3407, cuda=0, epochs=50, batch_size=64, lr=0.0001, weight_decay=0.05, optimizer='AdamW', clip_value=1, dropout=0.1, classifier='all_patch_reps', downstream_dataset='PhysioNet-MI', datasets_dir='/courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/data/physionet_mi/processed_average', num_of_classes=4, model_dir='/courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/results/physionet_mi_4class/seed_3407', num_workers=4, label_smoothing=0.1, multi_lr=True, frozen=False, use_pretrained_weights=True, foundation_dir='/courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/models/pretrained/cbramod/pretrained_weights.pth')


The downstream dataset is PhysioNet-MI
6300 1734 1803
9837


/courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod/models/model_for_physio.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.backbone.load_state_dic

Model(
  (backbone): CBraMod(
    (patch_embedding): PatchEmbedding(
      (positional_encoding): Sequential(
        (0): Conv2d(200, 200, kernel_size=(19, 7), stride=(1, 1), padding=(9, 3), groups=200)
      )
      (proj_in): Sequential(
        (0): Conv2d(1, 25, kernel_size=(1, 49), stride=(1, 25), padding=(0, 24))
        (1): GroupNorm(5, 25, eps=1e-05, affine=True)
        (2): GELU(approximate='none')
        (3): Conv2d(25, 25, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
        (4): GroupNorm(5, 25, eps=1e-05, affine=True)
        (5): GELU(approximate='none')
        (6): Conv2d(25, 25, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
        (7): GroupNorm(5, 25, eps=1e-05, affine=True)
        (8): GELU(approximate='none')
      )
      (spectral_proj): Sequential(
        (0): Linear(in_features=101, out_features=200, bias=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-11):

  0%|          | 0/99 [00:00<?, ?it/s]


... (283 outputs left out)

 90%|████████▉ | 26/29 [00:45<00:04,  1.52s/it]


 93%|█████████▎| 27/29 [00:46<00:02,  1.48s/it]


100%|██████████| 29/29 [00:48<00:00,  1.67s/it]
***************************Test results************************
Test Evaluation: acc: 0.63069, kappa: 0.50751, f1: 0.63085
[[310  25  72  44]
 [ 33 305  63  48]
 [ 52  75 258  65]
 [ 43  67  79 264]]


model save in /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/results/physionet_mi_4class/seed_3407/epoch32_acc_0.63069_kappa_0.50751_f1_0.63085.pth
Done!!!!!


## 8 · Test scores for every finished seed

Reads every `seed_*/train.log`, rewrites `test_scores.csv`, and prints mean ± SD once there are several seeds, the way the paper reports results. Only the summary is parsed here; the evaluator and saved weights are the authors'.

In [8]:
METRICS = re.compile(r'Test Evaluation: acc: ([0-9.]+), kappa: ([0-9.]+), f1: ([0-9.]+)')
rows = []
for log_file in sorted(RESULTS.glob('seed_*/train.log')):
    found = METRICS.findall(log_file.read_text())
    if not found:
        continue                                   # unfinished or failed run
    acc, kappa, f1 = map(float, found[-1])
    gpu_file = log_file.parent / 'gpu.txt'
    rows.append(dict(seed=log_file.parent.name[len('seed_'):],
                     gpu=gpu_file.read_text().strip() if gpu_file.is_file() else '',
                     balanced_accuracy=acc, kappa=kappa, weighted_f1=f1))
if not rows:
    print('No finished training runs yet.')
else:
    with (RESULTS / 'test_scores.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    (RESULTS / 'authors_commit.txt').write_text(OFFICIAL_COMMIT + '\n')
    for r in rows:
        print(f"seed {r['seed']} ({r['gpu']}): balanced acc {r['balanced_accuracy']:.5f} | "
              f"kappa {r['kappa']:.5f} | weighted F1 {r['weighted_f1']:.5f}")
    if len(rows) > 1:
        import statistics
        for k in ('balanced_accuracy', 'kappa', 'weighted_f1'):
            v = [r[k] for r in rows]
            print(f'{k}: {statistics.mean(v):.4f} ± {statistics.stdev(v):.4f} (n={len(v)})')
    print('Saved:', RESULTS / 'test_scores.csv')

seed 3407 (Tesla V100-SXM2-32GB): balanced acc 0.63069 | kappa 0.50751 | weighted F1 0.63085
Saved: /courses/CS7150.202710/students/pitta.a/cbramod-head-interp/CBraMod-MI/results/physionet_mi_4class/test_scores.csv


## Folder organization

```text
/courses/CS7150.202710/students/pitta.a/cbramod-head-interp/
├── cbramod-head-interp/          # your GitHub repo (git pull / git push here)
│   ├── README.md
│   ├── hpc/                      # setup_env.sh, run_notebook.sbatch
│   └── notebooks/                # this notebook + your other notebooks
├── CBraMod/                      # official authors' code, pinned commit
├── envs/cbramod/                 # conda env (built once)
├── logs/                         # logs of notebook jobs
└── CBraMod-MI/                   # data + results, never in git
    ├── eeg-motor-movementimagery-dataset-1.0.0.zip
    ├── data/physionet_mi/
    │   ├── files/S001/S001R04.edf ...
    │   └── processed_average/data.mdb
    ├── models/pretrained/cbramod/pretrained_weights.pth
    └── results/physionet_mi_4class/
        ├── test_scores.csv, authors_commit.txt
        └── seed_3407/  (train.log, gpu.txt, epoch..._acc_..._kappa_..._f1_....pth)
```

**Back up results.** Student folders under `/courses` are deleted after the course ends. Copy `CBraMod-MI/results` (and anything else you want to keep) to Google Drive with the same rclone remote you used to bring the data over.